# 🛵 ระบบตรวจจับการสวมหมวกนิรภัย (Smart Helmet Detection)
## 📓 สมุดงานสำหรับเทรนโมเดล YOLOv8 บน Google Colab (GPU)

สมุดงานนี้ออกแบบมาสำหรับเทรนโมเดลตรวจจับการสวมหมวกนิรภัยจากไฟล์ชุดข้อมูล **`lastdata.zip`** โดยมี 2 คลาส:
1. `with_helmet` (สวมหมวกนิรภัย)
2. `without_helmet` (ไม่สวมหมวกนิรภัย)

---
### ⚡ ขั้นตอนการใช้งานง่ายๆ:
1. **ตรวจสอบ GPU:** ไปที่เมนู `Runtime` -> `Change runtime type` -> ตรวจสอบว่าเลือก **`T4 GPU`**
2. **เชื่อมต่อ Google Drive:** รัน Cell ที่ 2 แล้วกดยืนยันการเชื่อมต่อ
3. **กดรันตามลำดับ:** กดปุ่ม Play (▶️) ทีละช่องจนจบ ระบบจะเทรนและบันทึกโมเดล `best.pt` กลับเข้า Google Drive ให้โดยอัตโนมัติ

### 1️⃣ ตรวจสอบการเปิดใช้งาน GPU

In [ ]:
# ตรวจสอบสถานะการเชื่อมต่อ GPU
!nvidia-smi

### 2️⃣ เชื่อมต่อ Google Drive และค้นหาไฟล์ `lastdata.zip`
> 💡 **ข้อสังเกตกรณีไฟล์อยู่ในโฟลเดอร์ "แชร์กับฉัน" (Shared with me):**
> หากโฟลเดอร์ **กันล็อคเด้ออ้าย** อยู่ในแถบ *"แชร์กับฉัน"* ให้คลิกขวาที่โฟลเดอร์ -> เลือก **"จัดระเบียบ (Organize)"** -> **"เพิ่มทางลัด (Add shortcut)"** -> เลือก **"ไดรฟ์ของฉัน (My Drive)"** เพื่อให้ Colab มองเห็นไฟล์ทันที

In [ ]:
import os
import shutil
import zipfile
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. ค้นหาตำแหน่งไฟล์ lastdata.zip โดยอัตโนมัติ
candidate_paths = [
    '/content/lastdata.zip',
    '/content/drive/MyDrive/lastdata.zip',
    '/content/drive/MyDrive/กันล็อคเด้ออ้าย/lastdata.zip',
    '/content/drive/Shareddrives/กันล็อคเด้ออ้าย/lastdata.zip'
]

zip_file_path = None
for path in candidate_paths:
    if os.path.exists(path):
        zip_file_path = path
        break

# หากไม่พบในจุดหลัก ให้ทำการ scan หาในไดรฟ์
if not zip_file_path:
    print("🔍 กำลังค้นหาไฟล์ lastdata.zip ภายใน Google Drive...")
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        if 'lastdata.zip' in files:
            zip_file_path = os.path.join(root, 'lastdata.zip')
            break

if zip_file_path and os.path.exists(zip_file_path):
    file_size_mb = os.path.getsize(zip_file_path) / (1024 * 1024)
    print(f"✅ พบไฟล์ Dataset สำเร็จ!")
    print(ff"📍 พาธ: {zip_file_path}")
    print(ff"📦 ขนาดไฟล์: {file_size_mb:.2f} MB")
else:
    raise FileNotFoundError("❌ ไม่พบไฟล์ lastdata.zip! กรุณาตรวจสอบว่ามีไฟล์นี้ในไดรฟ์ หรือคลิกขวาเพิ่มทางลัด (Add Shortcut) มาไว้ที่ My Drive")

### 3️⃣ แตกไฟล์ชุดข้อมูล (Extract Dataset) และเตรียม `data.yaml`

In [ ]:
import yaml

DATASET_DIR = '/content/dataset'
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"⏳ กำลังแตกไฟล์ {os.path.basename(zip_file_path)} ไปยัง {DATASET_DIR}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_DIR)
print("✅ แตกไฟล์ Dataset เรียบร้อย!")

# ตั้งค่า data.yaml ให้ใช้ Absolute Path ชัดเจน ป้องกันปัญหา Path คลาดเคลื่อน
train_img_dir = os.path.join(DATASET_DIR, 'train', 'images')
val_img_dir = os.path.join(DATASET_DIR, 'valid', 'images')
test_img_dir = os.path.join(DATASET_DIR, 'test', 'images')

yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
data_config = {
    'train': train_img_dir,
    'val': val_img_dir,
    'nc': 2,
    'names': ['with_helmet', 'without_helmet']
}
if os.path.exists(test_img_dir):
    data_config['test'] = test_img_dir

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("\n--- ข้อมูลการตั้งค่า data.yaml ---")
with open(yaml_path, 'r') as f:
    print(f.read())

train_count = len(os.listdir(train_img_dir)) if os.path.exists(train_img_dir) else 0
val_count = len(os.listdir(val_img_dir)) if os.path.exists(val_img_dir) else 0
print(f"📊 สรุปจำนวนรูปภาพสำหรับการเทรน:")
print(f"   - ชุดฝึกสอน (Train): {train_count} รูป")
print(f"   - ชุดประเมินผล (Valid): {val_count} รูป")

### 4️⃣ ติดตั้ง Ultralytics (YOLOv8)

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

### 5️⃣ เริ่มต้นกระบวนการฝึกสอน (Train Model)
- **โมเดลที่เลือก:** `yolov8n.pt` (รุ่น Nano: เบา เร็ว เหมาะกับ Raspberry Pi และคอมพิวเตอร์ทั่วไป)
- **Epochs:** 70 รอบ (พร้อมระบบ Early Stopping: หากค่า Loss นิ่งไม่พัฒนาต่อใน 15 รอบ ระบบจะหยุดอัตโนมัติเพื่อประหยัดเวลา)
- **Image Size:** 640
- **Batch Size:** 16

In [ ]:
from ultralytics import YOLO

# 1. โหลดโมเดลตั้งต้น YOLOv8 Nano
model = YOLO('yolov8n.pt')

# 2. เริ่มต้นเทรนโมเดล
results = model.train(
    data=yaml_path,
    epochs=70,
    patience=15,
    batch=16,
    imgsz=640,
    device=0,
    workers=4,
    project='/content/runs/train',
    name='helmet_yolov8n',
    save=True,
    plots=True
)
print("🎉 การฝึกสอนโมเดลเสร็จสมบูรณ์!")

### 6️⃣ แสดงผลลัพธ์กราฟการประเมิน (Training Results & Confusion Matrix)

In [ ]:
from IPython.display import Image, display

run_dir = '/content/runs/train/helmet_yolov8n'

# ดูกราฟผลการเทรนรวม (Loss, mAP50, mAP50-95, Precision, Recall)
results_plot = os.path.join(run_dir, 'results.png')
if os.path.exists(results_plot):
    print("📈 กราฟผลลัพธ์การฝึกสอน (Training Curves):")
    display(Image(filename=results_plot, width=900))

# ดู Confusion Matrix
cm_plot = os.path.join(run_dir, 'confusion_matrix.png')
if os.path.exists(cm_plot):
    print("🎯 ตารางจำแนกความแม่นยำ (Confusion Matrix):")
    display(Image(filename=cm_plot, width=700))

### 7️⃣ ทดสอบตรวจจับภาพจริง (Inference Test)

In [ ]:
import glob
import random

# โหลดน้ำหนัก best.pt ที่เทรนได้
best_weights = os.path.join(run_dir, 'weights', 'best.pt')
trained_model = YOLO(best_weights)

# สุ่มภาพจาก Valid Set จำนวน 3 ภาพมาทดสอบทำนายผล
val_images = glob.glob(f"{data_config['val']}/*.jpg") + glob.glob(f"{data_config['val']}/*.png")
if val_images:
    sample_images = random.sample(val_images, min(3, len(val_images)))
    trained_model.predict(
        source=sample_images,
        conf=0.35,
        save=True,
        project='/content/runs/predict',
        name='test_results',
        exist_ok=True
    )
    
    pred_dir = '/content/runs/predict/test_results'
    for img_p in glob.glob(f"{pred_dir}/*.jpg") + glob.glob(f"{pred_dir}/*.png"):
        print(f"🖼️ ผลการตรวจจับ: {os.path.basename(img_p)}")
        display(Image(filename=img_p, width=600))
else:
    print("ไม่พบภาพในชุด Valid สำหรับทดสอบ")

### 8️⃣ ส่งออกโมเดล และบันทึกผลลัพธ์กลับสู่ Google Drive ทันที

In [ ]:
# 1. ส่งออกโมเดลเป็นฟอร์แมต ONNX (สำหรับนำไปรันบนอุปกรณ์ประมวลผลขนาดเล็ก เช่น Raspberry Pi)
try:
    print("📦 กำลัง Export โมเดลเป็น ONNX...")
    onnx_file = trained_model.export(format='onnx')
    print(f"✅ Export ONNX สำเร็จ: {onnx_file}")
except Exception as e:
    print(f"⚠️ ข้ามขั้นตอน ONNX export: {e}")

# 2. กำหนดโฟลเดอร์ปลายทางใน Google Drive
drive_target_dir = '/content/drive/MyDrive/กันล็อคเด้ออ้าย'
if not os.path.exists(drive_target_dir):
    drive_target_dir = '/content/drive/MyDrive/helmet_model_trained'
os.makedirs(drive_target_dir, exist_ok=True)

# 3. คัดลอกไฟล์โมเดลและรายงานผล
files_to_copy = [
    (os.path.join(run_dir, 'weights', 'best.pt'), 'best.pt'),
    (os.path.join(run_dir, 'weights', 'last.pt'), 'last.pt'),
    (os.path.join(run_dir, 'results.png'), 'results.png'),
    (os.path.join(run_dir, 'confusion_matrix.png'), 'confusion_matrix.png')
]

print(f"\n💾 กำลังสำเนาผลลัพธ์ไปยัง Google Drive: {drive_target_dir} ...")
for src, filename in files_to_copy:
    if os.path.exists(src):
        dest = os.path.join(drive_target_dir, filename)
        shutil.copy(src, dest)
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        print(f"   ✔️ บันทึก {filename} ({size_mb:.2f} MB) สำเร็จ")

print("\n✨ บันทึกทุกอย่างลง Google Drive เรียบร้อย! คุณสามารถนำไฟล์ best.pt ไปใช้งานในเครื่องหรือ Raspberry Pi ได้เลย")